In [1]:
import os
import json
import time
import pandas as pd
import re
import asyncio
import nest_asyncio
nest_asyncio.apply()
from openai import AsyncOpenAI

# Place your OpenRouter API key in a file called 'API_key' in this directory

async_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="REDACTED-SEE-.env-AT-REPO-ROOT",
)

In [2]:
# DIALECT-COPA datasets
# Place the corresponding *-test.jsonl files in the datasets/ folder
# e.g. datasets/copa-en-test.jsonl, datasets/copa-sl-test.jsonl, etc.
tests = [
    "copa-en",
    "copa-sl",
    "copa-hr",
    "copa-hr-ckm",
    "copa-mk",
    "copa-sl-cer",
    "copa-sr",
    "copa-sr-tor",
    "copa-sl-prl"
]

# OpenRouter models
models = [
    "anthropic/claude-sonnet-4.6",
    "anthropic/claude-opus-4.6",
    "google/gemini-3.1-pro-preview",
    "google/gemini-3.1-flash-lite-preview",
    "openai/gpt-5.4-pro",
    "openai/gpt-5.4",
    "mistralai/mistral-large-2512",
    "mistralai/mistral-small-2603",
    "meta-llama/llama-4-maverick"
]

In [3]:
async def _predict_one(semaphore, entry, df_test_name, gpt_model, idx):
    if df_test_name == "copa-en":
        prompt = (
            'You will be given a task. The task definition is in English, '
            'as is the task itself. Here is the task!\n'
            f'Given the premise "{entry["premise"]}",'
        )
    else:
        prompt = (
            'You will be given a task. The task definition is in English, '
            'but the task itself is in another language. Here is the task!\n'
            f'Given the premise "{entry["premise"]}",'
        )

    if entry["question"] == "cause":
        prompt += " and that we are looking for the cause of this premise,"
    else:
        prompt += " and that we are looking for the result of this premise, "

    prompt += (
        f'which hypothesis is more plausible?\n'
        f'Hypothesis 1: "{entry["choice1"]}".\n'
        f'Hypothesis 2: "{entry["choice2"]}".\n\n'
        f'### Output format\n'
        f"Return a valid JSON dictionary with the following key: 'answer' "
        f"and a value should be an integer -- either 1 (if hypothesis 1 is "
        f"more plausible) or 2 (if hypothesis 2 is more plausible)."
    )

    async with semaphore:
        for attempt in range(5):
            try:
                kwargs = {
                    "model": gpt_model,
                    "messages": [{"role": "user", "content": prompt}],
                    "temperature": 0,
                }
                if "gemini" not in gpt_model.lower():
                    kwargs["response_format"] = {"type": "json_object"}
                completion = await async_client.chat.completions.create(**kwargs)
                if not completion.choices or completion.choices[0].message.content is None:
                    wait = 2 ** attempt * 5
                    print(f"idx={idx} empty response, retrying in {wait}s...")
                    await asyncio.sleep(wait)
                    continue
                raw = completion.choices[0].message.content
                match = re.search(r'"?\'?answer"?\'?\s*:\s*([12])', raw)
                if match:
                    return idx, int(match.group(1)) - 1
                else:
                    print(f"idx={idx} could not parse: {raw}")
                    return idx, -1
            except Exception as e:
                wait = 2 ** attempt * 5
                print(f"idx={idx} attempt {attempt+1}/5 failed: {e}. Retrying in {wait}s...")
                await asyncio.sleep(wait)
    return idx, -1


def predict_gpt(df_test_name, gpt_model, concurrency=20):
    os.makedirs("submissions", exist_ok=True)

    model_name = gpt_model.split("/")[1]
    out_path = f"submissions/submission-{model_name}-{df_test_name}.json"

    if os.path.exists(out_path):
        print(f"[skip] {out_path} already exists")
        return

    # Gemini chokes at high concurrency — throttle it
    if "gemini" in gpt_model.lower():
        concurrency = 5

    entries = [json.loads(line) for line in open(f"datasets/{df_test_name}-test.jsonl")]

    async def run_all():
        semaphore = asyncio.Semaphore(concurrency)
        tasks = [_predict_one(semaphore, entry, df_test_name, gpt_model, i)
                 for i, entry in enumerate(entries)]
        return await asyncio.gather(*tasks)

    start_time = time.time()
    results = asyncio.get_event_loop().run_until_complete(run_all())
    elapsed = time.time() - start_time

    responses = [pred for _, pred in sorted(results)]

    n = len(responses)
    print(f"Prediction finished. {elapsed/60:.2f} min for {n} instances — {elapsed/n:.3f} s/instance.")

    current_results = {
        "system": gpt_model,
        "predictions": [{"train": "NA (zero-shot)", "test": df_test_name, "predictions": responses}],
    }

    model_name = gpt_model.split("/")[1]
    out_path = f"submissions/submission-{model_name}-{df_test_name}.json"
    if os.path.exists(out_path):
        print(f"[skip] {out_path} already exists")
        return
    with open(out_path, "w") as f:
        json.dump(current_results, f)

    print(f"Saved: {out_path}")

In [4]:
# Run all models on all datasets
# Tip: comment out models or datasets you want to skip / re-run individually
for model in models:
    for test in tests:
        print(f"\n=== {model} | {test} ===")
        predict_gpt(test, model)


=== anthropic/claude-sonnet-4.6 | copa-en ===
[skip] submissions/submission-claude-sonnet-4.6-copa-en.json already exists

=== anthropic/claude-sonnet-4.6 | copa-sl ===
[skip] submissions/submission-claude-sonnet-4.6-copa-sl.json already exists

=== anthropic/claude-sonnet-4.6 | copa-hr ===
[skip] submissions/submission-claude-sonnet-4.6-copa-hr.json already exists

=== anthropic/claude-sonnet-4.6 | copa-hr-ckm ===
[skip] submissions/submission-claude-sonnet-4.6-copa-hr-ckm.json already exists

=== anthropic/claude-sonnet-4.6 | copa-mk ===
[skip] submissions/submission-claude-sonnet-4.6-copa-mk.json already exists

=== anthropic/claude-sonnet-4.6 | copa-sl-cer ===
[skip] submissions/submission-claude-sonnet-4.6-copa-sl-cer.json already exists

=== anthropic/claude-sonnet-4.6 | copa-sr ===
[skip] submissions/submission-claude-sonnet-4.6-copa-sr.json already exists

=== anthropic/claude-sonnet-4.6 | copa-sr-tor ===
[skip] submissions/submission-claude-sonnet-4.6-copa-sr-tor.json already